In [ ]:
!git clone https://github.com/Rithusravya/Emasters_Group-2_CapstoneProject.git

In [ ]:
!pip install -q transformers accelerate torch sentencepiece peft faiss-cpu
!pip install -q datasets evaluate nltk sqlparse rouge-score code-bert-score PyYAML matplotlib pandas

In [1]:
import os, sys

# REPO_ROOT = os.getcwd()
# if not os.path.isdir(os.path.join(REPO_ROOT, "src")):
#     # e.g. running on Colab: clone the repo if it isn't already checked out
#     if not os.path.isdir("Emasters_Group-2_CapstoneProject"):
#         !git clone -q https://github.com/Rithusravya/Emasters_Group-2_CapstoneProject.git
#     REPO_ROOT = os.path.abspath("Emasters_Group-2_CapstoneProject")
#     os.chdir(REPO_ROOT)
#
# sys.path.insert(0, REPO_ROOT)
# print("Repo root:", REPO_ROOT)

In [2]:
%matplotlib inline
import json
import logging
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - [%(levelname)s] - %(message)s")
logger = logging.getLogger("checkpoint_2")
from src.config_loader import CFG
from src.dataset_loader import UnifiedDatasetLoader
from src.models.tokenizer import CodeTokenizer
from src.models.ai_small_model import CodeGenModelWrapper
from src.generators.doc_generator import generate_docs_batch, pairs_from_codocbench
from src.evaluation.metrics import CodeMetricsEvaluator
from src.generators.sql_generator import evaluate_batch as evaluate_sql_batch
from src.generators.program_generator import synthesize_batch
from src.evaluation.comparator import ArchitectureComparator
from src.evaluation.visualization import ResultsVisualizer
from IPython.display import Image, display

'NoneType' object has no attribute 'cadam32bit_grad_fp32'


## 1. Load config, datasets, and the base model

In [4]:
datasets = UnifiedDatasetLoader.load_datasets(CFG)
for name, rows in datasets.items():
    print(
        f"{name:12s} -> {len(rows):4d} rows"
    )

2026-08-02 14:14:57,558 - [INFO] - Loading benchmarks and datasets from data/raw ...
2026-08-02 14:14:57,562 - [INFO] - Loaded 200 rows from data/raw/Spider (4 .jsonl file(s), 0 .json file(s))
2026-08-02 14:14:57,568 - [INFO] - Loaded 200 rows from data/raw/BirdBench (0 .jsonl file(s), 1 .json file(s))
2026-08-02 14:14:57,579 - [INFO] - Loaded 200 rows from data/raw/CoDocBench (3 .jsonl file(s), 0 .json file(s))


spider       ->  200 rows
birdbench    ->  200 rows
codocbench   ->  200 rows


In [5]:
model = CodeGenModelWrapper(
    model_name=CFG.model.name_or_path,
    device=CFG.project.device,
    max_length=CFG.model.max_length,
    use_lora=True,
    lora_kwargs=dict(CFG.lora),
)

print(
    "Loaded:",
    CFG.model.name_or_path,
    "on",
    model.device
)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359
Loaded: Qwen/Qwen2.5-Coder-0.5B-Instruct on mps


## 2. Documentation generation — BERTScore

In [6]:
N_DOC_SAMPLES = 20

doc_pairs = pairs_from_codocbench(
    datasets["codocbench"]
)[:N_DOC_SAMPLES]

if doc_pairs:
    doc_codes, doc_refs = zip(*doc_pairs)
    doc_codes = list(doc_codes)
    doc_refs = list(doc_refs)

    doc_predictions = generate_docs_batch(
        model,
        doc_codes,
    )

    doc_metrics = CodeMetricsEvaluator.evaluate_generation(
        doc_predictions,
        doc_refs,
        lang="python",
    )
else:
    doc_predictions = []
    doc_metrics = {
        "n":0,
        "bleu":0,
        "codebleu":0,
        "code_bertscore":0,
        "exact_match":0,
    }

print("\nDocumentation Generation")
print(
    "n:",
    doc_metrics["n"]
)
print(
    "BLEU:",
    round(doc_metrics["bleu"],4)
)
print(
    "BERTScore:",
    round(doc_metrics["code_bertscore"],4)
)

if doc_predictions:
    print("\nExample")
    print(
        "REFERENCE:",
        doc_refs[0]
    )
    print(
        "PRED:",
        doc_predictions[0]
    )

CodeBLEU error: Tree-sitter language for python not available. Please install the language parser using `pip install tree-sitter-python`.


AttributeError: type object 'CodeMetricsEvaluator' has no attribute 'calculate_bertscore'

## 3. Text-to-SQL generation

In [ ]:
N_SQL_SAMPLES = 20

def get_schema(row):

    """
    Extract schema if dataset provides it.

    Current loader does not provide schema,
    therefore this returns a descriptive fallback.
    """

    if row.get("schema"):
        return row["schema"]

    return f"""
Database name:
{row.get('db_id','unknown')}

No schema information available.

Infer only if absolutely necessary.
"""

sql_examples = []

for row in datasets["birdbench"][:N_SQL_SAMPLES]:
    if not row.get("question"):
        continue

    if not row.get("SQL"):
        continue

    sql_examples.append(
        {
            "schema":
                get_schema(row),

            "question":
                row["question"],

            "gold_sql":
                row["SQL"],

            # no execution without database
            "db_path":
                None,
        }
    )

sql_results = evaluate_sql_batch(
    model,
    sql_examples,
)


print("\nText-to-SQL")
print(
    "Samples:",
    sql_results["n"]
)

print(
    "Exact match:",
    round(
        sql_results["exact_match_acc"],
        4
    )
)

print(
    "Execution accuracy: unavailable"
)

if sql_results["predictions"]:
    print("\nExample")
    print(
        "QUESTION:",
        sql_examples[0]["question"]
    )
    print(
        "GOLD:",
        sql_examples[0]["gold_sql"]
    )
    print(
        "PRED:",
        sql_results["predictions"][0]
    )

## 4. Program generation

In [ ]:
program_tasks = [
    "Return the maximum value in a list of integers.",
    "Check whether a string is a palindrome.",
    "Compute the nth Fibonacci number.",
]

program_outputs = synthesize_batch(
    model,
    program_tasks,
)

print("\nProgram Generation")

for q,a in zip(
    program_tasks,
    program_outputs
):

    print(
        "\nProblem:",
        q
    )

    print(a)

## 5. Visualization

In [ ]:
results = {
    "Documentation Generation":
    {
        "n":
        doc_metrics["n"],

        "BLEU":
        doc_metrics["bleu"],

        "BERTScore":
        doc_metrics["code_bertscore"],

        "Accuracy":
        doc_metrics["exact_match"],
    },

    "Text-to-SQL":
    {
        "n":
        sql_results["n"],

        "Accuracy":
        sql_results["exact_match_acc"],

        "ExecutionAccuracy":
        None,
    }

}

os.makedirs(
    CFG.outputs.metrics_dir,
    exist_ok=True
)

with open(
    f"{CFG.outputs.metrics_dir}/checkpoint_2_results.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        results,
        f,
        indent=2
    )
print(
    "\nSaved metrics."
)

## 6. Comparator

In [ ]:
architecture_comparison = ArchitectureComparator.compare_architectures()
chart_path = ResultsVisualizer.generate_chart(architecture_comparison, CFG.outputs.plots_dir,
                                               filename="checkpoint_2_architecture_comparison.png")
display(Image(filename=chart_path))

In [ ]:
os.makedirs(CFG.outputs.metrics_dir, exist_ok=True)
ResultsVisualizer.save_metrics(results, f"{CFG.outputs.metrics_dir}/checkpoint_2_results.json")

fig.savefig(os.path.join(CFG.outputs.plots_dir, "checkpoint_2_accuracy_bertscore.png"), dpi=300)

print("Saved:")
print(" -", f"{CFG.outputs.metrics_dir}/checkpoint_2_results.json")
print(" -", os.path.join(CFG.outputs.plots_dir, "checkpoint_2_accuracy_bertscore.png"))
print(" -", chart_path)

In [3]:
import gc
import torch

for name in list(globals()):
    obj = globals()[name]
    if isinstance(obj, torch.Tensor):
        del globals()[name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

if torch.backends.mps.is_available():
    torch.mps.empty_cache()

print("All tensor memory cleared.")

All tensor memory cleared.
